# Funções de pertinência do fuzzy threshold

## Objetivo

Gera a figura das três funções de pertinência usadas pelo fuzzy threshold a partir dos percentis do conjunto de teste.

In [ ]:
# ruff: noqa: E402
import sys
from pathlib import Path

NOTEBOOK_CWD = Path.cwd().resolve()
for candidate in (NOTEBOOK_CWD, NOTEBOOK_CWD.parent, NOTEBOOK_CWD.parent.parent):
    src_dir = candidate / "src"
    if src_dir.exists():
        if str(src_dir) not in sys.path:
            sys.path.insert(0, str(src_dir))
        break

import json
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from utils.project.dataset import get_data_splits
from utils.project.notebook_env import (
    configure_notebook_environment,
    resolve_imagecas_base_path,
)
from utils.processing.preprocessing import downscale_image
from utils.utils.nifti_io import load_raw_img_and_label

REPO_ROOT = configure_notebook_environment(chdir_to_src=False)

CONFIG_PATH = REPO_ROOT / "config" / "pipeline_config.json"
OUTPUT_DIR = REPO_ROOT / "output" / "segmentation" / "analysis" / "fuzzy_membership_functions"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with CONFIG_PATH.open("r", encoding="utf-8") as file_handle:
    CONFIG = json.load(file_handle)

MIN_HU = float(CONFIG.get("MIN_THRESHOLD", -300))
DOWNSCALE_FACTORS = tuple(CONFIG.get("DOWNSCALE_FACTORS", [2, 2, 1]))
USE_OPENCV = CONFIG.get("DOWNSCALE_METHOD", "opencv") == "opencv"
OPENCV_INTERPOLATION = getattr(cv2, f"INTER_{CONFIG.get('OPENCV_INTERPOLATION', 'linear').upper()}", cv2.INTER_LINEAR)
FUZZY_CONFIG = CONFIG.get("THRESHOLDING", {}).get("fuzzy", {})
SOFT_MARGIN_HU = float(FUZZY_CONFIG.get("soft_margin_hu", 160))

OBJECT_PERCENTILE = 99.5
DENSE_PERCENTILE = 99.7
MAX_TEST_IMAGES = 60

BASE_PATH = resolve_imagecas_base_path()
print(f"Dataset: {BASE_PATH}")


## Análise

As subseções abaixo apresentam as métricas, tabelas ou visualizações do objetivo definido.

## Centros das classes

Para evitar empilhar todos os voxels do teste em memoria, os percentis sao calculados por imagem depois do downscale do pipeline e depois agregados pela media. O numero de imagens usadas e controlado por `MAX_TEST_IMAGES`.

## Configuração

Ajuste nesta seção apenas os parâmetros da análise; o pipeline base não é alterado.

## Carregamento

Carrega ou constrói os dados necessários para as análises seguintes.

In [ ]:
_, _, test_ids, _ = get_data_splits(str(BASE_PATH))
if MAX_TEST_IMAGES is not None:
    test_ids = test_ids[: int(MAX_TEST_IMAGES)]
print(f"Usando {len(test_ids)} imagens do conjunto de teste")
print(f"Downscale factors: {DOWNSCALE_FACTORS} | OpenCV: {USE_OPENCV}")

rows = []
for index, img_id in enumerate(test_ids, start=1):
    img_path = BASE_PATH / f"{img_id}.img.nii.gz"
    nii_img, _ = load_raw_img_and_label(str(img_path))
    volume = np.asarray(nii_img.get_fdata(dtype=np.float32), dtype=np.float32)
    down_volume = downscale_image(
        volume,
        DOWNSCALE_FACTORS,
        order=3,
        use_opencv=USE_OPENCV,
        opencv_interpolation=OPENCV_INTERPOLATION,
    )
    values = down_volume[np.isfinite(down_volume)]
    valid_values = values[values >= MIN_HU]
    if valid_values.size == 0:
        valid_values = values

    rows.append(
        {
            "IMG_ID": int(img_id),
            "p99_5_hu": float(np.percentile(valid_values, OBJECT_PERCENTILE)),
            "p99_7_hu": float(np.percentile(valid_values, DENSE_PERCENTILE)),
            "n_valid_voxels": int(valid_values.size),
        }
    )

    if index % 50 == 0 or index == len(test_ids):
        print(f"Processadas {index}/{len(test_ids)} imagens")

percentiles_df = pd.DataFrame(rows)
percentiles_df.head()

In [ ]:
soft_center = MIN_HU - SOFT_MARGIN_HU
object_center = float(percentiles_df["p99_5_hu"].mean())
dense_center = float(percentiles_df["p99_7_hu"].mean())

centers_df = pd.DataFrame(
    [
        {"classe": "fundo mole", "centro_hu": soft_center, "origem": f"MIN_HU - {SOFT_MARGIN_HU:g} HU"},
        {"classe": "objeto", "centro_hu": object_center, "origem": f"media do P{OBJECT_PERCENTILE:g} no teste"},
        {"classe": "fundo denso", "centro_hu": dense_center, "origem": f"media do P{DENSE_PERCENTILE:g} no teste"},
    ]
)
centers_df

## Figura 300 dpi

As curvas abaixo seguem a mesma logica implementada no pipeline: fundo mole decresce ate `MIN_HU`, fundo denso cresce entre o centro de objeto e o centro denso, e objeto fica alto na regiao intermediaria. Depois as pertinencias sao normalizadas para somarem 1 em cada HU.

In [ ]:
soft_width = max(MIN_HU - soft_center, np.finfo(np.float32).eps)
dense_width = max(dense_center - object_center, np.finfo(np.float32).eps)

x_min = soft_center - 80
x_max = dense_center + 80
hu_values = np.linspace(x_min, x_max, 1600, dtype=np.float32)

mu_soft = np.clip((MIN_HU - hu_values) / soft_width, 0.0, 1.0)
mu_dense = np.clip((hu_values - object_center) / dense_width, 0.0, 1.0)
mu_object = np.minimum(1.0 - mu_soft, 1.0 - mu_dense)

memberships = np.vstack([mu_soft, mu_object, mu_dense])
memberships = memberships / np.maximum(memberships.sum(axis=0, keepdims=True), np.finfo(np.float32).eps)
mu_soft_norm, mu_object_norm, mu_dense_norm = memberships

plt.rcParams.update(
    {
        "font.size": 16,
        "axes.labelsize": 18,
        "xtick.labelsize": 15,
        "ytick.labelsize": 15,
        "legend.fontsize": 15,
    }
)

fig, ax = plt.subplots(figsize=(10.5, 6.2), dpi=300)
ax.plot(hu_values, mu_soft_norm, linewidth=2.5, color="#2F6BFF", label="Fundo mole")
ax.plot(hu_values, mu_object_norm, linewidth=2.5, color="#1B9E77", label="Objeto")
ax.plot(hu_values, mu_dense_norm, linewidth=2.5, color="#D95F02", label="Fundo denso")

center_lines = [
    (soft_center, "Fundo mole", "#2F6BFF", (-52, 34)),
    (object_center, "Objeto", "#1B9E77", (-86, 78)),
    (dense_center, "Fundo denso", "#D95F02", (54, 18)),
]
for center, label, color, offset in center_lines:
    ax.axvline(center, linestyle="--", linewidth=1.6, color=color, alpha=0.8)
    ax.annotate(
        f"{label}\n{center:.1f} HU",
        xy=(center, 1.0),
        xytext=offset,
        textcoords="offset points",
        color=color,
        ha="center",
        va="bottom",
        fontsize=14,
        fontweight="bold",
        bbox={"boxstyle": "round,pad=0.25", "fc": "white", "ec": color, "alpha": 0.92},
        arrowprops={"arrowstyle": "-", "color": color, "lw": 1.2, "alpha": 0.85},
        clip_on=False,
    )

ax.set_xlabel("Intensidade (HU)", labelpad=10)
ax.set_ylabel("Pertinência", labelpad=10)
ax.set_ylim(-0.02, 1.30)
ax.set_xlim(x_min, x_max)
ax.grid(True, alpha=0.25, linewidth=0.8)
ax.legend(loc="lower left", frameon=True, framealpha=0.95, edgecolor="0.85")
fig.tight_layout()

figure_path = OUTPUT_DIR / "fuzzy_membership_functions_test_mean_p995_p997_300dpi.png"
fig.savefig(figure_path, dpi=300, bbox_inches="tight")
print(f"Figura salva em: {figure_path}")
plt.show()

In [ ]:
summary_path = OUTPUT_DIR / "fuzzy_membership_centers_test_mean_p995_p997.csv"
centers_df.to_csv(summary_path, index=False)
print(f"Resumo dos centros salvo em: {summary_path}")

## Conclusão

A figura e o CSV documentam os centros e as funções de pertinência usadas pelo fuzzy threshold para o conjunto analisado.